In [ ]:
# 1. Mount Google Drive
from google.colab import drive
import os
import shutil

print("📡 Step 1: Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=True)

# 2. Define the exact file paths
SOURCE_FOLDER = '/content/drive/MyDrive/Colab Notebooks'
PROJECT_FOLDER = '/content/drive/MyDrive/XAI_Energy_Forecasting'
FILE_NAME = 'explainable_consumption_project.ipynb'

source_file = os.path.join(SOURCE_FOLDER, FILE_NAME)
destination_file = os.path.join(PROJECT_FOLDER, FILE_NAME)

# 3. Perform the copy operation
print(f"\n📦 Step 2: Locating '{FILE_NAME}'...")
if os.path.exists(source_file):
    shutil.copy(source_file, destination_file)
    print(f"🎉 SUCCESS! A copy has been created in your project folder.")
    print(f"📁 Destination path: {destination_file}")
else:
    print(f"❌ Error: Could not find '{FILE_NAME}' in 'Colab Notebooks'.")
    print("💡 Tip: Make sure you went to File -> Save in the top menu bar first!")

📡 Step 1: Connecting to Google Drive...
Mounted at /content/drive

📦 Step 2: Locating 'explainable_consumption_project.ipynb'...
🎉 SUCCESS! A copy has been created in your project folder.
📁 Destination path: /content/drive/MyDrive/XAI_Energy_Forecasting/explainable_consumption_project.ipynb


In [2]:
import pandas as pd
import numpy as np

def clean_energy_data(file_path):
    print("⏳ Loading raw dataset...")
    # Load the data, treating '?' strings as actual missing NaN values
    df = pd.read_csv(file_path, sep=',', low_memory=False, na_values=['?'])

    print(f"📊 Initial shape: {df.shape}")
    print("🧹 Starting cleaning pipeline...")

    # 1. Combine Date and Time into a single datetime column
    # Adjust format syntax ('%d/%m/%Y' or '%Y-%m-%d') depending on your specific CSV structure
    df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], errors='coerce')

    # Drop the old separate Date and Time columns
    df = df.drop(columns=['Date', 'Time'])

    # Set Datetime as the index and sort chronologically
    df = df.set_index('Datetime')
    df = df.sort_index()

    # 2. Convert all structural columns to floats for numerical calculations
    cols_to_convert = [col for col in df.columns]
    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # 3. Handle Missing Values using Time-Series Forward Fill (ffill)
    # This carries the last known valid reading forward, preserving data continuity
    missing_count = df.isnull().sum().sum()
    if missing_count > 0:
        print(f"⚠️ Found {missing_count} missing values. Applying time-series forward fill...")
        df = df.ffill()
        # If any missing values remain at the very start, backward fill them
        df = df.bfill()
    else:
        print("✨ No missing values detected!")

    print(f"✅ Cleaning complete. Final shape: {df.shape}")
    return df

# --- RUNNING THE PIPELINE ---
# Update this path to match whichever CSV file you have inside your folder!
DATA_PATH = "/content/drive/MyDrive/XAI_Energy_Forecasting/household_power_consumption.txt"

try:
    cleaned_df = clean_energy_data(DATA_PATH)

    # Display the top rows of your clean dataset to make sure it looks perfect
    display(cleaned_df.head())

    # Save the clean dataframe as a new file in your folder so you don't have to clean it again
    OUTPUT_PATH = "/content/drive/MyDrive/XAI_Energy_Forecasting/cleaned_household_power_consumption.csv"
    cleaned_df.to_csv(OUTPUT_PATH)
    print(f"💾 Cleaned dataset successfully saved to: {OUTPUT_PATH}")

except Exception as e:
    print(f"❌ Pipeline failed: {e}")
    print("💡 Double-check your file path or separator if your dataset uses ';' instead of ','")

⏳ Loading raw dataset...
❌ Pipeline failed: 'utf-8' codec can't decode byte 0xeb in position 3: invalid continuation byte
💡 Double-check your file path or separator if your dataset uses ';' instead of ','
